What is LoRA / PEFT?
LoRA is a method where instead of updating all model weights, you inject and train small low-rank matrices into specific layers (e.g., attention, FFN).

PEFT is the umbrella framework that supports LoRA, prefix tuning, adapters, etc. via the 🤗 PEFT library.

Use when You're working with large base models (e.g., LLaMA, Mistral, Falcon, Gemma, etc.)

Steps:
1. Select a model
💡 For lightweight experiments: use "TinyLlama/TinyLlama-1.1B" or "microsoft/phi-2".

2. Prepare dataset:
2.1. Use JSONL format with instruction-style input: {"instruction": "Translate to French", "input": "I love you", "output": "Je t'aime"}

Example json file
{"instruction": "Write a 5-day powerlifting routine.", "input": "", "output": "..."}
{"instruction": "What muscles do squats target?", "input": "", "output": "..."}
{"instruction": "Compare sumo vs conventional deadlift.", "input": "", "output": "..."}


2.2. Unsupervised: (Language Modeling) Format -> Entire book
You don’t use instructions — you just teach the model to predict the next token. This is used with Causal Language Modeling (CLM) tasks — ideal if you just want the model to “know” the book.



I need A LoRA-compatible trainer (like Hugging Face peft + transformers).

3. Load Base Model and Tokenizer

4. Prepare LoRA Configuration

5. Tokenize Dataset

6. Train with trainer

7. Save model



In [1]:

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset

/home/mrosaria/Projects/NLP/GymRat/ratenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from huggingface_hub import login

login(token=""")


# Not working

In [7]:
# Load model & tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"#"mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [8]:
# Apply LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, peft_config)


In [ ]:
# Load and tokenize dataset
dataset = load_dataset("text", data_files={"train": "../data/book_test.txt"})
def tokenize(example): return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
tokenized_dataset = dataset.map(tokenize, batched=True)

Generating train split: 150 examples [00:00, 57424.75 examples/s]
Map: 100%|██████████| 150/150 [00:00<00:00, 5219.61 examples/s]


In [10]:
# Training config
training_args = TrainingArguments(
    output_dir="./lora-finetuned",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_dir="./logs",
    save_steps=500,
    save_total_limit=2,
    learning_rate=5e-5,
    fp16=True,
    report_to="none"
)

In [13]:

# Start training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)
#trainer.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


# NEW

In [5]:
dataset = load_dataset("text", data_files="../data/fine_tune_chunks.txt")

In [6]:
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [7]:
def tokenize(element):
    return tokenizer(element["text"])

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

block_size = 512  # or 1024, depending on model context length

Map: 100%|██████████| 13035/13035 [00:00<00:00, 54396.80 examples/s]


In [10]:
def group_texts(examples):
    # Concatenate texts
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    print('labels ', result["input_ids"])
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_dataset.map(group_texts, batched=True)

Map:  15%|█▌        | 2000/13035 [00:00<00:00, 12576.60 examples/s]

labels  [[1, 3824, 6369, 297, 29871, 29906, 29900, 29906, 29896, 491, 7229, 706, 350, 2152, 19088, 9266, 29889, 1, 14187, 1266, 29871, 30211, 29871, 29906, 29900, 29906, 29896, 319, 5022, 6912, 816, 335, 322, 4942, 29889, 19323, 5791, 386, 1648, 1, 2178, 10462, 21676, 1, 1939, 760, 310, 445, 17745, 1122, 367, 9483, 1133, 470, 13235, 1, 297, 738, 883, 470, 491, 738, 2794, 29892, 27758, 470, 28310, 29892, 470, 1, 6087, 297, 263, 2566, 470, 5663, 16837, 1788, 29892, 1728, 7536, 3971, 1, 10751, 515, 278, 9805, 261, 29889, 1, 2210, 29899, 29896, 29941, 29901, 29871, 29929, 29929, 29871, 29896, 29896, 29946, 29871, 29896, 29896, 29896, 29871, 29896, 29900, 29955, 29871, 29896, 29900, 29896, 29871, 29896, 29896, 29946, 29871, 29945, 29900, 29871, 29946, 29947, 29871, 29946, 29929, 29871, 29945, 29946, 1, 450, 2472, 5134, 297, 445, 3143, 338, 363, 28976, 1, 1, 11976, 871, 29889, 739, 338, 451, 9146, 470, 2411, 2957, 304, 367, 263, 23764, 1, 363, 10257, 16083, 9848, 29889, 450, 9591, 881, 2337,

Map:  46%|████▌     | 6000/13035 [00:00<00:00, 14024.58 examples/s]

labels  [[1, 1, 20685, 6669, 29892, 9817, 29892, 322, 22805, 453, 29892, 1346, 1523, 20941, 310, 652, 242, 175, 131, 261, 296, 1948, 292, 24472, 3476, 267, 30024, 313, 4149, 4443, 29871, 29947, 29929, 2038, 467, 1, 19155, 275, 5621, 322, 19135, 29892, 1346, 27107, 310, 1250, 6788, 297, 28563, 267, 30024, 313, 4149, 4443, 29871, 29896, 2038, 467, 1, 435, 29889, 678, 1772, 6669, 29875, 29892, 476, 29889, 2739, 20144, 29892, 319, 29889, 390, 1943, 8934, 29892, 341, 29889, 341, 29889, 6518, 29926, 19266, 29892, 322, 317, 29889, 341, 29889, 22805, 453, 29892, 1346, 29931, 398, 1646, 805, 457, 25806, 1, 508, 367, 18765, 287, 411, 385, 633, 3129, 979, 1339, 29873, 322, 29914, 272, 11664, 938, 336, 29899, 370, 3129, 979, 12959, 3995, 1, 7824, 1706, 457, 8237, 29871, 29947, 29892, 694, 29889, 29871, 29945, 313, 29896, 29929, 29929, 29929, 1125, 29871, 29941, 29947, 29947, 29994, 29929, 29945, 29889, 1, 435, 29889, 382, 29889, 3172, 261, 29892, 435, 29889, 390, 29889, 379, 870, 2330, 29892, 322,

Map:  92%|█████████▏| 12000/13035 [00:00<00:00, 14451.32 examples/s]

labels  [[1, 29871, 29953, 29936, 341, 29889, 12026, 29879, 3249, 538, 29892, 478, 29889, 476, 10471, 264, 29892, 349, 29889, 319, 7781, 538, 29892, 317, 29889, 1938, 404, 292, 29892, 349, 29889, 6971, 264, 29892, 319, 29889, 379, 29889, 997, 1295, 264, 29892, 405, 29889, 315, 29889, 476, 2741, 585, 29892, 1, 1, 341, 29889, 476, 1764, 261, 29892, 322, 317, 29889, 349, 29889, 19975, 375, 1100, 29892, 1346, 29907, 441, 293, 520, 1489, 333, 297, 24247, 29892, 16882, 296, 2200, 4845, 457, 10674, 271, 6694, 322, 1, 9416, 5232, 17711, 6694, 297, 2373, 514, 279, 10331, 262, 459, 493, 29891, 3995, 2522, 392, 262, 485, 713, 8237, 310, 27529, 669, 1, 9327, 297, 12453, 29871, 29896, 29929, 29892, 694, 29889, 29871, 29953, 313, 29906, 29900, 29900, 29929, 1125, 29871, 29955, 29929, 29900, 29994, 29947, 29900, 29906, 29889, 1, 319, 29889, 8075, 29892, 29871, 30222, 29889, 365, 713, 29892, 390, 29889, 350, 5610, 29892, 360, 29889, 319, 29889, 12321, 29892, 322, 478, 29889, 7073, 14642, 29892, 1346, 

Map: 100%|██████████| 13035/13035 [00:00<00:00, 14000.35 examples/s]

labels  []


In [13]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", load_in_8bit=True)
#The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [14]:
from peft import get_peft_model, LoraConfig, TaskType

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, config)


In [15]:
training_args = TrainingArguments(
    output_dir="./tinyllama-finetuned",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [16]:
trainer.train()


Step,Training Loss
10,3.401000
20,3.552500
30,3.295400
40,3.307400
50,3.373600
60,3.239900
70,3.082200
80,2.960500
90,2.790200
100,2.862200


TrainOutput(global_step=573, training_loss=2.6280408429849835, metrics={'train_runtime': 207.5136, 'train_samples_per_second': 5.523, 'train_steps_per_second': 2.761, 'total_flos': 3645978766737408.0, 'train_loss': 2.6280408429849835, 'epoch': 3.0})

In [19]:
trainer.save_model("tinyllama-book-finetuned")
tokenizer.save_pretrained("tinyllama-book-finetuned")


('tinyllama-book-finetuned/tokenizer_config.json',
 'tinyllama-book-finetuned/special_tokens_map.json',
 'tinyllama-book-finetuned/chat_template.jinja',
 'tinyllama-book-finetuned/tokenizer.json')

In [21]:
from transformers import pipeline


In [25]:
generator = pipeline("text-generation", model="tinyllama-book-finetuned", tokenizer=tokenizer)


Device set to use cuda:0


In [28]:
print(generator("Forcing someone with hip anteversion to lift with a technique requires?", max_new_tokens=200)[0]['generated_text'])


Forcing someone with hip anteversion to lift with a technique requires?


In [29]:

generator = pipeline("text-generation", model="tinyllama-book-finetuned", tokenizer=tokenizer)
print(generator("What is the importance of proper recovery in powerlifting?", max_new_tokens=100)[0]['generated_text'])


Device set to use cuda:0


What is the importance of proper recovery in powerlifting?
